[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/08_cicd.ipynb)

*Engineering Toolbox · Notebook 8 of 8*

# CI/CD With GitHub Actions

Automate everything after git push: matrix builds, job graphs and critical paths, path and branch filters, cache keys, concurrency, secret masking, step conditions, check status, artifacts, workflow validation, schedules in UTC, CI minutes, test sharding, reruns, changelogs, image pushing rules, environment promotion and DORA metrics.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** CI is an automatic inspector at the factory door: nothing leaves until it passes every check.

**What interviewers really test here:** What a pipeline should check, speeding it up, caching and matrices, secrets and least privilege, deploying through environments and rolling back.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Your first CI workflow *(Basics)*
- [ ] 2. Fast pipelines *(Core)*
- [ ] 3. Continuous delivery *(Core)*
- [ ] 4. Securing the pipeline *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### CI, continuous delivery, continuous deployment

| Practice | Means |
| --- | --- |
| Continuous integration | every change is merged often and built and tested automatically |
| Continuous delivery | every passing change is ready to release with one click |
| Continuous deployment | every passing change goes to production automatically |

### DORA metrics

| Metric | Elite teams |
| --- | --- |
| Deployment frequency | on demand, many times a day |
| Lead time for changes | less than a day |
| Change failure rate | low, single-digit to ~15% |
| Time to restore service | less than an hour |

---
# Part 2: Lessons, step by step

## Lesson 1: Your first CI workflow

*Level: Basics*

**Continuous integration** runs your checks on every push and pull request, so a broken change is caught in minutes, not in production. In GitHub Actions a **workflow** (a YAML file in `.github/workflows/`) is triggered by **events** and contains **jobs**; each job runs **steps** on a fresh runner machine.



```python
name: ci
on:
  pull_request:
  push: {branches: [main]}
concurrency: {group: "ci-${{ github.ref }}", cancel-in-progress: true}
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix: {python: ["3.11", "3.12"]}
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5
        with: {python-version: "${{ matrix.python }}", enable-cache: true}
      - run: uv sync --frozen
      - run: uv run ruff check . && uv run ruff format --check .
      - run: uv run mypy app
      - run: uv run pytest -q --cov=app
      - if: failure()
        run: echo "tests failed on ${{ matrix.python }}"
```

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q14](#q14), [Q2](#q2), [Q15](#q15), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 2: Fast pipelines

*Level: Core*

- Run independent jobs in **parallel**; only chain with `needs:` when one really depends on another. The **critical path** sets the minimum time.
- **Cache** dependencies keyed on the lockfile hash, with a restore-key fallback.
- Use **path filters** so docs changes don't run the whole test suite.
- **Shard** long test suites across parallel jobs; **cancel** superseded runs; re-run only failed jobs.
- Watch cost: macOS and Windows minutes are multiplied.



```python
jobs:
  test:
    strategy: {matrix: {shard: [1, 2, 3]}}
    steps:
      - run: uv run pytest --splits 3 --group ${{ matrix.shard }}   # pytest-split
  build:
    needs: [test]
    ...
      - uses: actions/upload-artifact@v4
        with: {name: wheel, path: dist/}
```

**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q13](#q13), [Q16](#q16), [Q5](#q5), [Q4](#q4), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 3: Continuous delivery

*Level: Core*

Once checks pass, the pipeline builds an image, tags it, pushes it to a registry and deploys it — first to **dev**, then **staging**, then **production** after tests and, often, a manual approval (GitHub **environments** support required reviewers). Releases get version tags and a changelog generated from commit messages.



```python
  release:
    if: startsWith(github.ref, 'refs/tags/v')
    runs-on: ubuntu-latest
    environment: production          # waits for approval
    permissions: {contents: write, packages: write}
    steps:
      - uses: actions/checkout@v4
      - uses: docker/login-action@v3
        with: {registry: ghcr.io, username: "${{ github.actor }}", password: "${{ secrets.GITHUB_TOKEN }}"}
      - uses: docker/build-push-action@v6
        with: {push: true, tags: "ghcr.io/acme/api:${{ github.ref_name }}"}
```

**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q18](#q18), [Q17](#q17), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 4: Securing the pipeline

*Level: Advanced*

- Store credentials in **secrets** (masked in logs) — never in the YAML. Prefer short-lived **OIDC** credentials for cloud deploys over long-lived keys.
- Give the `GITHUB_TOKEN` **least-privilege** `permissions:` per job.
- **Pin** third-party actions to commit SHAs, and let Dependabot update them.
- Don't run untrusted pull-request code with access to secrets (`pull_request_target` is dangerous).

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q8](#q8), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import re

from collections import defaultdict


<a id="q1"></a>
### Q1. Mask Secrets in Logs

🟢 Warm-up · *\*\*\* in the job output*

GitHub replaces every registered secret value in log output with `***`. Mask all occurrences of each secret (longest secrets first, so a secret containing another is fully hidden). Return the masked lines and how many replacements were made.

**Examples**

- `lines = ['Logging in to registry with token ghp_ABC123xyz', 'export DB_URL=postgres://app:Pa55word@db/prod', 'password is Pa55word, again Pa55word', 'all clean here'], secrets = ['ghp_ABC123xyz', 'Pa55word', 'postgres://app:Pa55word@db/prod']` → `(['Logging in to registry with token ***', 'export DB_URL=***', 'password is ***, again ***', 'all clean here'], 4)`

In [ ]:
def mask_secrets(lines, secrets):
    """
    GitHub replaces every registered secret value in log output with ***. Mask
    all occurrences of each secret (longest secrets first, so a secret
    containing another is fully hidden). Return the masked lines and how many
    replacements were made.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mask_secrets, [
    ((['Logging in to registry with token ghp_ABC123xyz', 'export DB_URL=postgres://app:Pa55word@db/prod', 'password is Pa55word, again Pa55word', 'all clean here'], ['ghp_ABC123xyz', 'Pa55word', 'postgres://app:Pa55word@db/prod']), (['Logging in to registry with token ***', 'export DB_URL=***', 'password is ***, again ***', 'all clean here'], 4)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with str.count and replace.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One alternation regex with subn.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with str.count and replace.

```python
def mask_secrets(lines, secrets):
    order = sorted(secrets, key=len, reverse=True)
    out, n = [], 0
    for line in lines:
        for s in order:
            n += line.count(s)
            line = line.replace(s, "***")
        out.append(line)
    return (out, n)
```

**Cost:** O(n · s · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One alternation regex with subn.

```python
def mask_secrets(lines, secrets):
    rx = re.compile("|".join(re.escape(s) for s in sorted(secrets, key=len, reverse=True)))
    out, n = [], 0
    for line in lines:
        line, k = rx.subn("***", line)
        out.append(line)
        n += k
    return (out, n)
```

**Cost:** O(n · L) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with str.count and replace. That costs O(n · s · L) time and O(L) space."

"I can do better: One alternation regex with subn. That's O(n · L) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/mask-secrets-in-logs)

<a id="q2"></a>
### Q2. Can the Pull Request Merge Yet?

🟢 Warm-up · *Combined check status*

Given check runs `(name, status)` where status is `queued`, `in_progress`, `success`, `failure`, `skipped` or `cancelled`, and the `required` checks: return `'failure'` if any required check failed or was cancelled, `'pending'` if any required check is missing or not finished, else `'success'` (skipped counts as passing).

**Examples**

- `checks = [('lint', 'success'), ('tests (3.12)', 'in_progress'), ('tests (3.11)', 'success'), ('docs', 'failure')], required = ['lint', 'tests (3.12)', 'tests (3.11)']` → `'pending'`
- `checks = [('lint', 'success'), ('tests', 'cancelled')], required = ['lint', 'tests']` → `'failure'`
- `checks = [('lint', 'success'), ('deploy-preview', 'skipped')], required = ['lint', 'deploy-preview', 'security']` → `'pending'`

In [ ]:
def combined_status(checks, required):
    """
    Given check runs (name, status) where status is queued, in_progress,
    success, failure, skipped or cancelled, and the required checks: return
    'failure' if any required check failed or was cancelled, 'pending' if any
    required check is missing or not finished, else 'success' (skipped counts as
    passing).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(combined_status, [
    (([('lint', 'success'), ('tests (3.12)', 'in_progress'), ('tests (3.11)', 'success'), ('docs', 'failure')], ['lint', 'tests (3.12)', 'tests (3.11)']), 'pending'),
    (([('lint', 'success'), ('tests', 'cancelled')], ['lint', 'tests']), 'failure'),
    (([('lint', 'success'), ('deploy-preview', 'skipped')], ['lint', 'deploy-preview', 'security']), 'pending'),
    (([('lint', 'success'), ('tests', 'skipped')], ['lint', 'tests']), 'success'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over required checks with flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A status dict and any().

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over required checks with flags.

```python
def combined_status(checks, required):
    status = {}
    for n, s in checks:
        status[n] = s
    pending = False
    for r in required:
        s = status.get(r)
        if s in ("failure", "cancelled"):
            return "failure"
        if s not in ("success", "skipped"):
            pending = True
    return "pending" if pending else "success"
```

**Cost:** O(r + c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A status dict and any().

```python
def combined_status(checks, required):
    st = dict(checks)
    got = [st.get(r) for r in required]
    if any(s in ("failure", "cancelled") for s in got):
        return "failure"
    return "success" if all(s in ("success", "skipped") for s in got) else "pending"
```

**Cost:** O(r + c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over required checks with flags. That costs O(r + c) time and O(c) space."

"I can do better: A status dict and any(). That's O(r + c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/can-the-pull-request-merge-yet)

<a id="q3"></a>
### Q3. Scheduled Workflows Run in UTC

🟢 Warm-up · *Convert cron times*

GitHub Actions cron schedules are in UTC. For each `'minute hour'` pair, return the local time `'HH:MM'` for a UTC offset in minutes (e.g. +330 for IST), plus whether it lands on the previous, same or next day (`-1`, `0`, `+1`).

**Examples**

- `schedules = ['30 3', '0 0', '45 20', '15 23'], offset = 330` → `[('09:00', 0), ('05:30', 0), ('02:15', 1), ('04:45', 1)]`
- `schedules = ['0 2', '30 23'], offset = -300` → `[('21:00', -1), ('18:30', 0)]`

In [ ]:
def utc_to_local(schedules, offset):
    """
    GitHub Actions cron schedules are in UTC. For each 'minute hour' pair,
    return the local time 'HH:MM' for a UTC offset in minutes (e.g. +330 for
    IST), plus whether it lands on the previous, same or next day (-1, 0, +1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(utc_to_local, [
    ((['30 3', '0 0', '45 20', '15 23'], 330), [('09:00', 0), ('05:30', 0), ('02:15', 1), ('04:45', 1)]),
    ((['0 2', '30 23'], -300), [('21:00', -1), ('18:30', 0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add minutes and fix overflow with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

divmod on total minutes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Add minutes and fix overflow with if-statements.

```python
def utc_to_local(schedules, offset):
    out = []
    for s in schedules:
        m, h = map(int, s.split())
        total = h * 60 + m + offset
        day = 0
        if total < 0:
            total += 1440
            day = -1
        elif total >= 1440:
            total -= 1440
            day = 1
        out.append((f"{total // 60:02d}:{total % 60:02d}", day))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** divmod on total minutes.

```python
def utc_to_local(schedules, offset):
    out = []
    for s in schedules:
        m, h = map(int, s.split())
        day, mins = divmod(h * 60 + m + offset, 1440)
        out.append((f"{mins // 60:02d}:{mins % 60:02d}", day))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add minutes and fix overflow with if-statements. That costs O(n) time and O(n) space."

"I can do better: divmod on total minutes. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/scheduled-workflows-run-in-utc)

<a id="q4"></a>
### Q4. What Does CI Cost?

🟢 Warm-up · *Runner minutes*

Jobs are `(runner, minutes)` per run. Each job's minutes round **up** to whole minutes, then multiply by the runner's multiplier (Linux 1, Windows 2, macOS 10). Given runs per month and free minutes, return `(billable minutes per month, cost at price per minute rounded to 2)`.

**Examples**

- `jobs = [('linux', 3.2), ('linux', 7.9), ('windows', 6.1), ('macos', 11.5)], runs = 400, free = 3000, price = 0.008` → `(55400, 443.2)`

In [ ]:
MULTIPLIER = {"linux": 1, "windows": 2, "macos": 10}

def ci_cost(jobs, runs, free, price):
    """
    Jobs are (runner, minutes) per run. Each job's minutes round up to whole
    minutes, then multiply by the runner's multiplier (Linux 1, Windows 2, macOS
    10). Given runs per month and free minutes, return (billable minutes per
    month, cost at price per minute rounded to 2).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ci_cost, [
    (([('linux', 3.2), ('linux', 7.9), ('windows', 6.1), ('macos', 11.5)], 400, 3000, 0.008), (55400, 443.2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with math.ceil.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

sum() over a generator.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with math.ceil.

```python
def ci_cost(jobs, runs, free, price):
    per_run = 0
    for runner, mins in jobs:
        per_run += math.ceil(mins) * MULTIPLIER[runner]
    billable = per_run * runs - free
    if billable < 0:
        billable = 0
    return (billable, round(billable * price, 2))
```

**Cost:** O(j) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** sum() over a generator.

```python
def ci_cost(jobs, runs, free, price):
    billable = max(0, runs * sum(math.ceil(m) * MULTIPLIER[r] for r, m in jobs) - free)
    return (billable, round(billable * price, 2))
```

**Cost:** O(j) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with math.ceil. That costs O(j) time and O(1) space."

"I can do better: sum() over a generator. That's O(j) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/what-does-ci-cost)

<a id="q5"></a>
### Q5. Re-run Only Failed Jobs

🟢 Warm-up · *Save time on flaky failures*

A pipeline ran jobs `(name, minutes, passed)`. Re-running the whole workflow costs every job again; re-running failed jobs costs only those. Return `(minutes for a full re-run, minutes for failed-only, minutes saved)`.

**Examples**

- `jobs = [('lint', 1, True), ('unit', 6, True), ('integration', 14, False), ('e2e', 22, True), ('build', 9, False)]` → `(52, 23, 29)`

In [ ]:
def rerun_cost(jobs):
    """
    A pipeline ran jobs (name, minutes, passed). Re-running the whole workflow
    costs every job again; re-running failed jobs costs only those. Return
    (minutes for a full re-run, minutes for failed-only, minutes saved).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rerun_cost, [
    (([('lint', 1, True), ('unit', 6, True), ('integration', 14, False), ('e2e', 22, True), ('build', 9, False)],), (52, 23, 29)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Two running totals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two sum() calls.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Two running totals.

```python
def rerun_cost(jobs):
    full = failed = 0
    for _, m, ok in jobs:
        full += m
        if not ok:
            failed += m
    return (full, failed, full - failed)
```

**Cost:** O(j) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Two sum() calls.

```python
def rerun_cost(jobs):
    full, failed = sum(m for _, m, _ in jobs), sum(m for _, m, ok in jobs if not ok)
    return (full, failed, full - failed)
```

**Cost:** O(j) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Two running totals. That costs O(j) time and O(1) space."

"I can do better: Two sum() calls. That's O(j) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/re-run-only-failed-jobs)

<a id="q6"></a>
### Q6. Should This Run Push an Image?

🟢 Warm-up · *Build everywhere, publish carefully*

For each event `(event, ref)`: pull requests build but never push; a push to `refs/heads/main` pushes tags `main` and `latest`; a tag push `refs/tags/vX.Y.Z` pushes `X.Y.Z`; other branches push nothing. Return the list of tags pushed for each event.

**Examples**

- `events = [('pull_request', 'refs/pull/42/merge'), ('push', 'refs/heads/main'), ('push', 'refs/tags/v1.4.2'), ('push', 'refs/heads/feature/x'), ('push', 'refs/tags/nightly')]` → `[[], ['main', 'latest'], ['1.4.2'], [], []]`

In [ ]:
def push_tags(events):
    """
    For each event (event, ref): pull requests build but never push; a push to
    refs/heads/main pushes tags main and latest; a tag push refs/tags/vX.Y.Z
    pushes X.Y.Z; other branches push nothing. Return the list of tags pushed
    for each event.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(push_tags, [
    (([('pull_request', 'refs/pull/42/merge'), ('push', 'refs/heads/main'), ('push', 'refs/tags/v1.4.2'), ('push', 'refs/heads/feature/x'), ('push', 'refs/tags/nightly')],), [[], ['main', 'latest'], ['1.4.2'], [], []]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

If/elif per event.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex for version tags.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** If/elif per event.

```python
def push_tags(events):
    out = []
    for event, ref in events:
        if event == "pull_request":
            out.append([])
        elif ref == "refs/heads/main":
            out.append(["main", "latest"])
        elif ref.startswith("refs/tags/v") and all(p.isdigit() for p in ref[len("refs/tags/v"):].split(".")) and ref.count(".") == 2:
            out.append([ref[len("refs/tags/v"):]])
        else:
            out.append([])
    return out
```

**Cost:** O(e) time, O(e) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex for version tags.

```python
def push_tags(events):
    def tags(event, ref):
        if event != "push":
            return []
        if ref == "refs/heads/main":
            return ["main", "latest"]
        m = re.fullmatch(r"refs/tags/v(\d+\.\d+\.\d+)", ref)
        return [m[1]] if m else []
    return [tags(e, r) for e, r in events]
```

**Cost:** O(e) time, O(e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: If/elif per event. That costs O(e) time and O(e) space."

"I can do better: A regex for version tags. That's O(e) time and O(e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/should-this-run-push-an-image)

<a id="q7"></a>
### Q7. Pass Files Between Jobs

🟢 Warm-up · *upload-artifact / download-artifact*

Jobs run in the listed order; each uploads some artifacts and downloads others. A download fails if no **earlier** job uploaded that artifact. Return sorted `(job, missing artifact)` errors and the artifacts uploaded but never downloaded.

**Examples**

- `jobs = [('build', ['wheel', 'sbom'], []), ('test', ['coverage'], ['wheel']), ('docker', ['image-digest'], ['wheel', 'frontend-dist']), ('deploy', [], ['image-digest', 'coverage-badge'])]` → `([('deploy', 'coverage-badge'), ('docker', 'frontend-dist')], ['coverage', 'sbom'])`

In [ ]:
def artifacts(jobs):
    """
    Jobs run in the listed order; each uploads some artifacts and downloads
    others. A download fails if no earlier job uploaded that artifact. Return
    sorted (job, missing artifact) errors and the artifacts uploaded but never
    downloaded.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(artifacts, [
    (([('build', ['wheel', 'sbom'], []), ('test', ['coverage'], ['wheel']), ('docker', ['image-digest'], ['wheel', 'frontend-dist']), ('deploy', [], ['image-digest', 'coverage-badge'])],), ([('deploy', 'coverage-badge'), ('docker', 'frontend-dist')], ['coverage', 'sbom'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Lists searched in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sets of uploaded and downloaded names.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Lists searched in a loop.

```python
def artifacts(jobs):
    uploaded, downloaded, errors = [], [], []
    for name, ups, downs in jobs:
        for d in downs:
            if d not in uploaded:
                errors.append((name, d))
            downloaded.append(d)
        uploaded += ups
    unused = sorted(u for u in uploaded if u not in downloaded)
    return (sorted(errors), unused)
```

**Cost:** O(j · a²) time, O(a) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Sets of uploaded and downloaded names.

```python
def artifacts(jobs):
    up, down, errors = set(), set(), []
    for name, ups, downs in jobs:
        errors += [(name, d) for d in downs if d not in up]
        down |= set(downs)
        up |= set(ups)
    return (sorted(errors), sorted(up - down))
```

**Cost:** O(j · a) time, O(a) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Lists searched in a loop. That costs O(j · a²) time and O(a) space."

"I can do better: Sets of uploaded and downloaded names. That's O(j · a) time and O(a) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/pass-files-between-jobs)

<a id="q8"></a>
### Q8. Pin Third-Party Actions

🟢 Warm-up · *Supply-chain safety*

A tag like `@v4` can be moved by whoever controls the action, so security-conscious teams pin third-party actions to a full 40-character commit SHA. Return the `uses:` values that aren't pinned, ignoring local actions (starting with `./`) and Docker images (starting with `docker://`).

**Examples**

- `uses = ['actions/checkout@v4', 'astral-sh/setup-uv@b5f58b2abc5763ade55e4e9d0fe52cd1ff7979ca', './.github/actions/notify', 'docker://alpine:3.20', 'softprops/action-gh-release@master', 'aws-actions/configure-aws-credentials@v4.0.2']` → `['actions/checkout@v4', 'softprops/action-gh-release@master', 'aws-actions/configure-aws-credentials@v4.0.2']`

In [ ]:
def unpinned_actions(uses):
    """
    A tag like @v4 can be moved by whoever controls the action, so security-
    conscious teams pin third-party actions to a full 40-character commit SHA.
    Return the uses: values that aren't pinned, ignoring local actions (starting
    with ./) and Docker images (starting with docker://).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(unpinned_actions, [
    ((['actions/checkout@v4', 'astral-sh/setup-uv@b5f58b2abc5763ade55e4e9d0fe52cd1ff7979ca', './.github/actions/notify', 'docker://alpine:3.20', 'softprops/action-gh-release@master', 'aws-actions/configure-aws-credentials@v4.0.2'],), ['actions/checkout@v4', 'softprops/action-gh-release@master', 'aws-actions/configure-aws-credentials@v4.0.2']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each part with string methods.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex for a 40-hex SHA.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Check each part with string methods.

```python
def unpinned_actions(uses):
    out = []
    for u in uses:
        if u.startswith("./") or u.startswith("docker://"):
            continue
        ref = u.split("@")[-1]
        if not (len(ref) == 40 and all(c in "0123456789abcdef" for c in ref)):
            out.append(u)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex for a 40-hex SHA.

```python
def unpinned_actions(uses):
    return [u for u in uses if not u.startswith(("./", "docker://")) and not re.search(r"@[0-9a-f]{40}$", u)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each part with string methods. That costs O(n) time and O(n) space."

"I can do better: A regex for a 40-hex SHA. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/pin-third-party-actions)

<a id="q9"></a>
### Q9. Expand a Build Matrix

🟡 Interview standard · *strategy.matrix*

Expand the matrix axes (in key order, first key varying slowest) into jobs, remove combinations matching every key of any `exclude` entry, then apply `include` entries: if an include's matrix keys match existing jobs, add its extra keys to those jobs; otherwise append it as a new job. Return the job dicts.

**Examples**

- `axes = {'os': ['ubuntu', 'windows'], 'python': ['3.11', '3.12']}, exclude = [{'os': 'windows', 'python': '3.11'}], include = [{'os': 'ubuntu', 'python': '3.12', 'coverage': True}, {'os': 'macos', 'python': '3.12'}]` → `[{'os': 'ubuntu', 'python': '3.11'}, {'os': 'ubuntu', 'python': '3.12', 'coverage': True}, {'os': 'windows', 'python': '3.12'}, {'os': 'macos', 'python': '3.12'}]`

In [ ]:
import itertools

def expand_matrix(axes, exclude, include):
    """
    Expand the matrix axes (in key order, first key varying slowest) into jobs,
    remove combinations matching every key of any exclude entry, then apply
    include entries: if an include's matrix keys match existing jobs, add its
    extra keys to those jobs; otherwise append it as a new job. Return the job
    dicts.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(expand_matrix, [
    (({'os': ['ubuntu', 'windows'], 'python': ['3.11', '3.12']}, [{'os': 'windows', 'python': '3.11'}], [{'os': 'ubuntu', 'python': '3.12', 'coverage': True}, {'os': 'macos', 'python': '3.12'}]), [{'os': 'ubuntu', 'python': '3.11'}, {'os': 'ubuntu', 'python': '3.12', 'coverage': True}, {'os': 'windows', 'python': '3.12'}, {'os': 'macos', 'python': '3.12'}]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops building dicts.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.product` and comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops building dicts.

```python
def expand_matrix(axes, exclude, include):
    jobs = [{}]
    for key, values in axes.items():
        new = []
        for j in jobs:
            for v in values:
                d = dict(j)
                d[key] = v
                new.append(d)
        jobs = new
    kept = []
    for j in jobs:
        drop = False
        for e in exclude:
            if all(j.get(k) == v for k, v in e.items()):
                drop = True
        if not drop:
            kept.append(j)
    for inc in include:
        core = {k: v for k, v in inc.items() if k in axes}
        matched = False
        for j in kept:
            if all(j[k] == v for k, v in core.items()):
                j.update(inc)
                matched = True
        if not matched:
            kept.append(dict(inc))
    return kept
```

**Cost:** O(product · (e + i)) time, O(product) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.product` and comprehensions.

```python
def expand_matrix(axes, exclude, include):
    jobs = [dict(zip(axes, c)) for c in itertools.product(*axes.values())]
    jobs = [j for j in jobs if not any(all(j.get(k) == v for k, v in e.items()) for e in exclude)]
    for inc in include:
        hits = [j for j in jobs if all(j[k] == v for k, v in inc.items() if k in axes)]
        for j in hits:
            j.update(inc)
        if not hits:
            jobs.append(dict(inc))
    return jobs
```

**Cost:** O(product · (e + i)) time, O(product) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops building dicts. That costs O(product · (e + i)) time and O(product) space."

"I can do better: `itertools.product` and comprehensions. That's O(product · (e + i)) time and O(product) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/expand-a-build-matrix)

<a id="q10"></a>
### Q10. Job Waves and the Critical Path

🟡 Interview standard · *needs:*

Jobs run in parallel unless they `need` other jobs. Return `(waves of jobs that can run together, each sorted, minimum pipeline minutes with unlimited runners)`. A job starts when all its needs have finished.

**Examples**

- `needs = {'lint': [], 'unit': [], 'build': ['lint'], 'integration': ['build', 'unit'], 'e2e': ['build'], 'deploy': ['integration', 'e2e']}, minutes = {'lint': 1, 'unit': 4, 'build': 6, 'integration': 8, 'e2e': 12, 'deploy': 3}` → `([['lint', 'unit'], ['build'], ['e2e', 'integration'], ['deploy']], 22)`

In [ ]:
def job_waves(needs, minutes):
    """
    Jobs run in parallel unless they need other jobs. Return (waves of jobs that
    can run together, each sorted, minimum pipeline minutes with unlimited
    runners). A job starts when all its needs have finished.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(job_waves, [
    (({'lint': [], 'unit': [], 'build': ['lint'], 'integration': ['build', 'unit'], 'e2e': ['build'], 'deploy': ['integration', 'e2e']}, {'lint': 1, 'unit': 4, 'build': 6, 'integration': 8, 'e2e': 12, 'deploy': 3}), ([['lint', 'unit'], ['build'], ['e2e', 'integration'], ['deploy']], 22)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly collect jobs whose needs are done.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Memoised finish times; waves by longest chain of needs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Repeatedly collect jobs whose needs are done.

```python
def job_waves(needs, minutes):
    done, waves = set(), []
    while len(done) < len(needs):
        wave = sorted(j for j in needs if j not in done and all(n in done for n in needs[j]))
        waves.append(wave)
        done |= set(wave)
    finish = {}
    for wave in waves:
        for j in wave:
            finish[j] = max([finish[n] for n in needs[j]], default=0) + minutes[j]
    return (waves, max(finish.values()))
```

**Cost:** O(j²) time, O(j) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Memoised finish times; waves by longest chain of needs.

```python
def job_waves(needs, minutes):
    from functools import lru_cache
    @lru_cache(None)
    def level(j):
        return 1 + max(map(level, needs[j]), default=-1)
    @lru_cache(None)
    def finish(j):
        return minutes[j] + max(map(finish, needs[j]), default=0)
    waves = defaultdict(list)
    for j in needs:
        waves[level(j)].append(j)
    return ([sorted(waves[i]) for i in sorted(waves)], max(map(finish, needs)))
```

**Cost:** O(j + e) time, O(j) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly collect jobs whose needs are done. That costs O(j²) time and O(j) space."

"I can do better: Memoised finish times; waves by longest chain of needs. That's O(j + e) time and O(j) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `needs = {'a': [], 'b': ['a']}, minutes = {'a': 1, 'b': 2}`

1. level() returns 0
2. level() returns 1
3. finish() returns 1
4. finish() returns 3
5. job\_waves() returns ([['a'], ['b']], 3)

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/job-waves-and-the-critical-path)

<a id="q11"></a>
### Q11. Which Workflows Run for This Change?

🟡 Interview standard · *on.push.paths*

Each workflow has `paths` (run only if some changed file matches; empty means any) and `paths_ignore` (don't run if every changed file matches). `**` matches across folders (`**/` also matches no folder at all), `*` within one (use `path_match`). For each change set, return the workflows that run, sorted.

**Examples**

- `workflows = {'backend': {'paths': ['app/**', 'pyproject.toml'], 'paths_ignore': []}, 'frontend': {'paths': ['web/**'], 'paths_ignore': []}, 'docs': {'paths': ['docs/**', '*.md'], 'paths_ignore': []}, 'ci': {'paths': [], 'paths_ignore': ['**/*.md', 'docs/**']}}, changes = [['app/api/orders.py'], ['README.md'], ['web/src/App.tsx', 'docs/guide.md'], ['pyproject.toml', 'uv.lock']]` → `[['backend', 'ci'], ['docs'], ['ci', 'docs', 'frontend'], ['backend', 'ci']]`

In [ ]:
def triggered(workflows, changes):
    """
    Each workflow has paths (run only if some changed file matches; empty means
    any) and paths_ignore (don't run if every changed file matches). ** matches
    across folders (**/ also matches no folder at all), * within one (use
    path_match). For each change set, return the workflows that run, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(triggered, [
    (({'backend': {'paths': ['app/**', 'pyproject.toml'], 'paths_ignore': []}, 'frontend': {'paths': ['web/**'], 'paths_ignore': []}, 'docs': {'paths': ['docs/**', '*.md'], 'paths_ignore': []}, 'ci': {'paths': [], 'paths_ignore': ['**/*.md', 'docs/**']}}, [['app/api/orders.py'], ['README.md'], ['web/src/App.tsx', 'docs/guide.md'], ['pyproject.toml', 'uv.lock']]), [['backend', 'ci'], ['docs'], ['ci', 'docs', 'frontend'], ['backend', 'ci']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops over workflows, files and patterns.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

any()/all() comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops over workflows, files and patterns.

```python
def path_match(pattern, path):
    rx = "".join({"**/": "(?:.*/)?", "**": ".*", "*": "[^/]*"}.get(t, re.escape(t)) for t in re.split(r"(\*\*/|\*\*|\*)", pattern))
    return re.fullmatch(rx, path) is not None


def triggered(workflows, changes):
    out = []
    for files in changes:
        run = []
        for name, wf in workflows.items():
            ok = True
            if wf["paths"]:
                ok = False
                for f in files:
                    for p in wf["paths"]:
                        if path_match(p, f):
                            ok = True
            if ok and wf["paths_ignore"]:
                every = True
                for f in files:
                    if not any(path_match(p, f) for p in wf["paths_ignore"]):
                        every = False
                if every:
                    ok = False
            if ok:
                run.append(name)
        out.append(sorted(run))
    return out
```

**Cost:** O(c · w · f · p) time, O(w) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** any()/all() comprehensions.

```python
def path_match(pattern, path):
    rx = "".join({"**/": "(?:.*/)?", "**": ".*", "*": "[^/]*"}.get(t, re.escape(t)) for t in re.split(r"(\*\*/|\*\*|\*)", pattern))
    return re.fullmatch(rx, path) is not None


def triggered(workflows, changes):
    hit = lambda pats, f: any(path_match(p, f) for p in pats)
    def runs(wf, files):
        wanted = not wf["paths"] or any(hit(wf["paths"], f) for f in files)
        ignored = bool(wf["paths_ignore"]) and all(hit(wf["paths_ignore"], f) for f in files)
        return wanted and not ignored
    return [sorted(n for n, wf in workflows.items() if runs(wf, files)) for files in changes]
```

**Cost:** O(c · w · f · p) time, O(w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops over workflows, files and patterns. That costs O(c · w · f · p) time and O(w) space."

"I can do better: any()/all() comprehensions. That's O(c · w · f · p) time and O(w) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `workflows = {'x': {'paths': ['a/**'], 'paths_ignore': []}}, changes = [['a/b.py']]`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. runs() returns True
4. triggered() returns [['x']]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/which-workflows-run-for-this-change)

<a id="q12"></a>
### Q12. Cache Hits, Partial Hits and Misses

🟡 Interview standard · *actions/cache keys*

The cache key is `'{os}-py{python}-{lock hash}'` and the restore key is the prefix `'{os}-py{python}-'`. For each run, return `'hit'` if an exact key is cached, `'partial'` if some cached key starts with the restore prefix (an older cache is restored, then updated), else `'miss'`. After a run that wasn't a hit, its exact key is saved.

**Examples**

- `runs = [('ubuntu', '3.12', 'a1'), ('ubuntu', '3.12', 'a1'), ('ubuntu', '3.12', 'b7'), ('ubuntu', '3.11', 'b7'), ('macos', '3.12', 'b7'), ('ubuntu', '3.12', 'a1')]` → `['miss', 'hit', 'partial', 'miss', 'miss', 'hit']`

In [ ]:
def cache_results(runs):
    """
    The cache key is '{os}-py{python}-{lock hash}' and the restore key is the
    prefix '{os}-py{python}-'. For each run, return 'hit' if an exact key is
    cached, 'partial' if some cached key starts with the restore prefix (an
    older cache is restored, then updated), else 'miss'. After a run that wasn't
    a hit, its exact key is saved.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cache_results, [
    (([('ubuntu', '3.12', 'a1'), ('ubuntu', '3.12', 'a1'), ('ubuntu', '3.12', 'b7'), ('ubuntu', '3.11', 'b7'), ('macos', '3.12', 'b7'), ('ubuntu', '3.12', 'a1')],), ['miss', 'hit', 'partial', 'miss', 'miss', 'hit']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of saved keys, scanned each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of keys and a set of prefixes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of saved keys, scanned each time.

```python
def cache_results(runs):
    saved, out = [], []
    for os_, py, h in runs:
        key, prefix = f"{os_}-py{py}-{h}", f"{os_}-py{py}-"
        if key in saved:
            out.append("hit")
            continue
        out.append("partial" if any(k.startswith(prefix) for k in saved) else "miss")
        saved.append(key)
    return out
```

**Cost:** O(r²) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of keys and a set of prefixes.

```python
def cache_results(runs):
    keys, prefixes, out = set(), set(), []
    for os_, py, h in runs:
        key, prefix = f"{os_}-py{py}-{h}", f"{os_}-py{py}-"
        out.append("hit" if key in keys else "partial" if prefix in prefixes else "miss")
        keys.add(key)
        prefixes.add(prefix)
    return out
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of saved keys, scanned each time. That costs O(r²) time and O(r) space."

"I can do better: A set of keys and a set of prefixes. That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/cache-hits-partial-hits-and-misses)

<a id="q13"></a>
### Q13. Cancel Superseded Runs

🟡 Interview standard · *concurrency: cancel-in-progress*

Runs are `(id, start minute, end minute, group)` sorted by start. With `cancel-in-progress`, a new run in a group cancels a still-running earlier run of the same group (one that hasn't ended before the new run starts). Return `(cancelled ids, runner-minutes saved)` — e.g. pushing three commits quickly to one PR builds only the last.

**Examples**

- `runs = [('r1', 0, 12, 'pr-7'), ('r2', 3, 15, 'pr-7'), ('r3', 4, 14, 'main'), ('r4', 5, 17, 'pr-7'), ('r5', 20, 30, 'pr-7'), ('r6', 21, 28, 'main')]` → `(['r1', 'r2'], 19)`

In [ ]:
def cancelled_runs(runs):
    """
    Runs are (id, start minute, end minute, group) sorted by start. With cancel-
    in-progress, a new run in a group cancels a still-running earlier run of the
    same group (one that hasn't ended before the new run starts). Return
    (cancelled ids, runner-minutes saved) — e.g. pushing three commits quickly
    to one PR builds only the last.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cancelled_runs, [
    (([('r1', 0, 12, 'pr-7'), ('r2', 3, 15, 'pr-7'), ('r3', 4, 14, 'main'), ('r4', 5, 17, 'pr-7'), ('r5', 20, 30, 'pr-7'), ('r6', 21, 28, 'main')],), (['r1', 'r2'], 19)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the running run per group.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with dict.get and a walrus.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the running run per group.

```python
def cancelled_runs(runs):
    current, cancelled, saved = {}, [], 0
    for rid, start, end, group in runs:
        if group in current:
            pid, pend = current[group]
            if pend > start:
                cancelled.append(pid)
                saved += pend - start
        current[group] = (rid, end)
    return (cancelled, saved)
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with dict.get and a walrus.

```python
def cancelled_runs(runs):
    current, cancelled, saved = {}, [], 0
    for rid, start, end, group in runs:
        if (prev := current.get(group)) and prev[1] > start:
            cancelled.append(prev[0])
            saved += prev[1] - start
        current[group] = (rid, end)
    return (cancelled, saved)
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the running run per group. That costs O(n) time and O(g) space."

"I can do better: Same with dict.get and a walrus. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/cancel-superseded-runs)

<a id="q14"></a>
### Q14. Which Steps Run?

🟡 Interview standard · *if: success(), failure(), always()*

Steps run in order with a condition: `'success'` (default: run only if nothing before failed), `'failure'` (run only if something before failed), `'always'`. `outcome` says whether a step would pass when run. Return `(steps that ran, job result)` — the job fails if any step that ran failed.

**Examples**

- `steps = [('checkout', 'success', True), ('install', 'success', True), ('test', 'success', False), ('coverage report', 'success', True), ('upload test results', 'always', True), ('notify on failure', 'failure', True)]` → `(['checkout', 'install', 'test', 'upload test results', 'notify on failure'], 'failure')`
- `steps = [('checkout', 'success', True), ('test', 'success', True), ('notify on failure', 'failure', True), ('cleanup', 'always', True)]` → `(['checkout', 'test', 'cleanup'], 'success')`

In [ ]:
def steps_run(steps):
    """
    Steps run in order with a condition: 'success' (default: run only if nothing
    before failed), 'failure' (run only if something before failed), 'always'.
    outcome says whether a step would pass when run. Return (steps that ran, job
    result) — the job fails if any step that ran failed.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(steps_run, [
    (([('checkout', 'success', True), ('install', 'success', True), ('test', 'success', False), ('coverage report', 'success', True), ('upload test results', 'always', True), ('notify on failure', 'failure', True)],), (['checkout', 'install', 'test', 'upload test results', 'notify on failure'], 'failure')),
    (([('checkout', 'success', True), ('test', 'success', True), ('notify on failure', 'failure', True), ('cleanup', 'always', True)],), (['checkout', 'test', 'cleanup'], 'success')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A failed flag and if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A condition table keyed by name.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A failed flag and if/elif.

```python
def steps_run(steps):
    failed, ran = False, []
    for name, cond, ok in steps:
        if cond == "always" or (cond == "success" and not failed) or (cond == "failure" and failed):
            ran.append(name)
            if not ok:
                failed = True
    return (ran, "failure" if failed else "success")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A condition table keyed by name.

```python
def steps_run(steps):
    failed, ran = False, []
    for name, cond, ok in steps:
        if {"always": True, "success": not failed, "failure": failed}[cond]:
            ran.append(name)
            failed |= not ok
    return (ran, "failure" if failed else "success")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A failed flag and if/elif. That costs O(n) time and O(n) space."

"I can do better: A condition table keyed by name. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/which-steps-run)

<a id="q15"></a>
### Q15. Validate a Workflow

🟡 Interview standard · *Catch mistakes before pushing*

Each job has `runs_on`, `needs` and `steps`. Return sorted errors: `'JOB: missing runs-on'`, `'JOB: no steps'`, `'JOB: needs unknown job X'`, and `'cycle: A, B, …'` (the jobs in any dependency cycle, sorted) if the needs graph has a cycle.

**Examples**

- `jobs = {'lint': {'runs_on': 'ubuntu-latest', 'needs': [], 'steps': ['ruff']}, 'test': {'runs_on': 'ubuntu-latest', 'needs': ['lint', 'deploy'], 'steps': ['pytest']}, 'build': {'runs_on': '', 'needs': ['lint', 'package'], 'steps': []}, 'deploy': {'runs_on': 'ubuntu-latest', 'needs': ['test'], 'steps': ['kubectl']}}` → `['build: missing runs-on', 'build: needs unknown job package', 'build: no steps', 'cycle: deploy, test']`

In [ ]:
def validate_workflow(jobs):
    """
    Each job has runs_on, needs and steps. Return sorted errors: 'JOB: missing
    runs-on', 'JOB: no steps', 'JOB: needs unknown job X', and 'cycle: A, B, …'
    (the jobs in any dependency cycle, sorted) if the needs graph has a cycle.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(validate_workflow, [
    (({'lint': {'runs_on': 'ubuntu-latest', 'needs': [], 'steps': ['ruff']}, 'test': {'runs_on': 'ubuntu-latest', 'needs': ['lint', 'deploy'], 'steps': ['pytest']}, 'build': {'runs_on': '', 'needs': ['lint', 'package'], 'steps': []}, 'deploy': {'runs_on': 'ubuntu-latest', 'needs': ['test'], 'steps': ['kubectl']}},), ['build: missing runs-on', 'build: needs unknown job package', 'build: no steps', 'cycle: deploy, test']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Checks in a loop; peel off jobs with no remaining needs to find a cycle.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Kahn's algorithm over known needs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Checks in a loop; peel off jobs with no remaining needs to find a cycle.

```python
def validate_workflow(jobs):
    errors = []
    for name, j in jobs.items():
        if not j["runs_on"]:
            errors.append(f"{name}: missing runs-on")
        if not j["steps"]:
            errors.append(f"{name}: no steps")
        for n in j["needs"]:
            if n not in jobs:
                errors.append(f"{name}: needs unknown job {n}")
    left = {n: [d for d in j["needs"] if d in jobs] for n, j in jobs.items()}
    changed = True
    while changed:
        changed = False
        for n in list(left):
            if all(d not in left for d in left[n]):
                del left[n]
                changed = True
    if left:
        errors.append("cycle: " + ", ".join(sorted(left)))
    return sorted(errors)
```

**Cost:** O(j²) time, O(j) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Kahn's algorithm over known needs.

```python
def validate_workflow(jobs):
    errors = [f"{n}: missing runs-on" for n, j in jobs.items() if not j["runs_on"]]
    errors += [f"{n}: no steps" for n, j in jobs.items() if not j["steps"]]
    errors += [f"{n}: needs unknown job {d}" for n, j in jobs.items() for d in j["needs"] if d not in jobs]
    indeg = {n: sum(d in jobs for d in j["needs"]) for n, j in jobs.items()}
    users = defaultdict(list)
    for n, j in jobs.items():
        for d in j["needs"]:
            if d in jobs:
                users[d].append(n)
    queue = [n for n, k in indeg.items() if k == 0]
    while queue:
        for u in users[queue.pop()]:
            indeg[u] -= 1
            if indeg[u] == 0:
                queue.append(u)
    stuck = sorted(n for n, k in indeg.items() if k > 0)
    return sorted(errors + ([f"cycle: {', '.join(stuck)}"] if stuck else []))
```

**Cost:** O(j + e) time, O(j) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Checks in a loop; peel off jobs with no remaining needs to find a cycle. That costs O(j²) time and O(j) space."

"I can do better: Kahn's algorithm over known needs. That's O(j + e) time and O(j) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/validate-a-workflow)

<a id="q16"></a>
### Q16. Split Tests Across Shards

🟡 Interview standard · *Parallel test jobs*

Split test files (with their durations in seconds) across `n` parallel jobs to minimise the slowest job: sort longest first and give each file to the currently least-loaded shard (ties: lowest shard number). Return `(shard totals, slowest shard seconds)`.

**Examples**

- `durations = {'test_api.py': 210, 'test_db.py': 340, 'test_cli.py': 30, 'test_models.py': 95, 'test_auth.py': 160, 'test_reports.py': 280, 'test_utils.py': 15, 'test_e2e.py': 410}, n = 3` → `([505, 515, 520], 520)`

In [ ]:
import heapq

def shard_tests(durations, n):
    """
    Split test files (with their durations in seconds) across n parallel jobs to
    minimise the slowest job: sort longest first and give each file to the
    currently least-loaded shard (ties: lowest shard number). Return (shard
    totals, slowest shard seconds).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shard_tests, [
    (({'test_api.py': 210, 'test_db.py': 340, 'test_cli.py': 30, 'test_models.py': 95, 'test_auth.py': 160, 'test_reports.py': 280, 'test_utils.py': 15, 'test_e2e.py': 410}, 3), ([505, 515, 520], 520)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan all shards for the minimum each time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A min-heap of (load, shard).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan all shards for the minimum each time.

```python
def shard_tests(durations, n):
    loads = [0] * n
    for _, d in sorted(durations.items(), key=lambda x: (-x[1], x[0])):
        i = loads.index(min(loads))
        loads[i] += d
    return (loads, max(loads))
```

**Cost:** O(f · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A min-heap of (load, shard).

```python
def shard_tests(durations, n):
    heap, loads = [(0, i) for i in range(n)], [0] * n
    for _, d in sorted(durations.items(), key=lambda x: (-x[1], x[0])):
        load, i = heapq.heappop(heap)
        loads[i] = load + d
        heapq.heappush(heap, (loads[i], i))
    return (loads, max(loads))
```

**Cost:** O(f log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan all shards for the minimum each time. That costs O(f · n) time and O(n) space."

"I can do better: A min-heap of (load, shard). That's O(f log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/split-tests-across-shards)

<a id="q17"></a>
### Q17. Generate a Changelog

🟡 Interview standard · *From conventional commits*

Group commit subjects into changelog sections in this order: `Breaking changes` (any `!` before the colon), `Features` (feat), `Fixes` (fix, perf); other types are left out. Return markdown lines: `'### Section'` followed by `'- description (scope)'` items (scope only if present), sections without items omitted.

**Examples**

- `commits = ['feat(api): add bulk export', 'fix: handle empty cart', 'chore: bump deps', 'feat!: drop Python 3.10', 'perf(db): batch inserts', 'docs: fix typo', 'fix(auth): refresh token race']` → `['### Breaking changes', '- drop Python 3.10', '### Features', '- add bulk export (api)', '### Fixes', '- handle empty cart', '- batch inserts (db)', '- refresh token race (auth)']`

In [ ]:
def changelog(commits):
    """
    Group commit subjects into changelog sections in this order: Breaking
    changes (any ! before the colon), Features (feat), Fixes (fix, perf); other
    types are left out. Return markdown lines: '### Section' followed by '-
    description (scope)' items (scope only if present), sections without items
    omitted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(changelog, [
    ((['feat(api): add bulk export', 'fix: handle empty cart', 'chore: bump deps', 'feat!: drop Python 3.10', 'perf(db): batch inserts', 'docs: fix typo', 'fix(auth): refresh token race'],), ['### Breaking changes', '- drop Python 3.10', '### Features', '- add bulk export (api)', '### Fixes', '- handle empty cart', '- batch inserts (db)', '- refresh token race (auth)']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Parse each subject and append to section lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A regex and a section lookup.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Parse each subject and append to section lists.

```python
def changelog(commits):
    sections = {"Breaking changes": [], "Features": [], "Fixes": []}
    for c in commits:
        head, _, desc = c.partition(": ")
        breaking = head.endswith("!")
        head = head.rstrip("!")
        typ, _, scope = head.partition("(")
        scope = scope.rstrip(")")
        item = "- " + desc + (f" ({scope})" if scope else "")
        if breaking:
            sections["Breaking changes"].append(item)
        elif typ == "feat":
            sections["Features"].append(item)
        elif typ in ("fix", "perf"):
            sections["Fixes"].append(item)
    out = []
    for title, items in sections.items():
        if items:
            out.append("### " + title)
            out += items
    return out
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A regex and a section lookup.

```python
def changelog(commits):
    rx = re.compile(r"^(\w+)(?:\(([^)]*)\))?(!)?: (.*)$")
    sections = {"Breaking changes": [], "Features": [], "Fixes": []}
    for c in commits:
        typ, scope, bang, desc = rx.match(c).groups()
        title = "Breaking changes" if bang else {"feat": "Features", "fix": "Fixes", "perf": "Fixes"}.get(typ)
        if title:
            sections[title].append(f"- {desc}" + (f" ({scope})" if scope else ""))
    return [line for t, items in sections.items() if items for line in [f"### {t}", *items]]
```

**Cost:** O(c) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Parse each subject and append to section lists. That costs O(c) time and O(c) space."

"I can do better: A regex and a section lookup. That's O(c) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/generate-a-changelog)

<a id="q18"></a>
### Q18. Promote Through Environments

🟡 Interview standard · *dev → staging → prod*

A build is deployed to `dev` if CI passed, promoted to `staging` if dev's smoke tests passed, and to `prod` only if staging tests passed **and** it was approved. Each build is a dict of booleans `ci`, `dev_smoke`, `staging_tests`, `approved`. Return the furthest environment each build reaches (or `'none'`).

**Examples**

- `builds = [{'ci': True, 'dev_smoke': True, 'staging_tests': True, 'approved': True}, {'ci': True, 'dev_smoke': True, 'staging_tests': True, 'approved': False}, {'ci': True, 'dev_smoke': False, 'staging_tests': True, 'approved': True}, {'ci': False, 'dev_smoke': True, 'staging_tests': True, 'approved': True}]` → `['prod', 'staging', 'dev', 'none']`

In [ ]:
def promotion(builds):
    """
    A build is deployed to dev if CI passed, promoted to staging if dev's smoke
    tests passed, and to prod only if staging tests passed and it was approved.
    Each build is a dict of booleans ci, dev_smoke, staging_tests, approved.
    Return the furthest environment each build reaches (or 'none').
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(promotion, [
    (([{'ci': True, 'dev_smoke': True, 'staging_tests': True, 'approved': True}, {'ci': True, 'dev_smoke': True, 'staging_tests': True, 'approved': False}, {'ci': True, 'dev_smoke': False, 'staging_tests': True, 'approved': True}, {'ci': False, 'dev_smoke': True, 'staging_tests': True, 'approved': True}],), ['prod', 'staging', 'dev', 'none']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk a list of (environment, gate) pairs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested if-statements.

```python
def promotion(builds):
    out = []
    for b in builds:
        env = "none"
        if b["ci"]:
            env = "dev"
            if b["dev_smoke"]:
                env = "staging"
                if b["staging_tests"] and b["approved"]:
                    env = "prod"
        out.append(env)
    return out
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Walk a list of (environment, gate) pairs.

```python
def promotion(builds):
    gates = [("dev", lambda b: b["ci"]), ("staging", lambda b: b["dev_smoke"]), ("prod", lambda b: b["staging_tests"] and b["approved"])]
    out = []
    for b in builds:
        env = "none"
        for name, ok in gates:
            if not ok(b):
                break
            env = name
        out.append(env)
    return out
```

**Cost:** O(b) time, O(b) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested if-statements. That costs O(b) time and O(b) space."

"I can do better: Walk a list of (environment, gate) pairs. That's O(b) time and O(b) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/promote-through-environments)

<a id="q19"></a>
### Q19. Measure Delivery: DORA Metrics

🟡 Interview standard · *How good is your delivery pipeline?*

Deployments are `(deploy hour, commit hour, failed, restored hour or None)` over `days` days. Return `{deploys_per_day (rounded to 2), lead_time_hours (median of deploy − commit, rounded to 1), change_failure_pct (rounded to 1), mttr_hours (mean of restored − deploy for failures, rounded to 1)}` — the four DORA metrics.

**Examples**

- `deploys = [(10, 2, False, None), (34, 30, True, 36.5), (58, 41, False, None), (80, 79, False, None), (105, 96, True, 109.0), (130, 120, False, None), (150, 149, False, None)], days = 7` → `{'deploys_per_day': 1.0, 'lead_time_hours': 8, 'change_failure_pct': 28.6, 'mttr_hours': 3.2}`

In [ ]:
import statistics

def dora(deploys, days):
    """
    Deployments are (deploy hour, commit hour, failed, restored hour or None)
    over days days. Return {deploys_per_day (rounded to 2), lead_time_hours
    (median of deploy − commit, rounded to 1), change_failure_pct (rounded to
    1), mttr_hours (mean of restored − deploy for failures, rounded to 1)} — the
    four DORA metrics.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dora, [
    (([(10, 2, False, None), (34, 30, True, 36.5), (58, 41, False, None), (80, 79, False, None), (105, 96, True, 109.0), (130, 120, False, None), (150, 149, False, None)], 7), {'deploys_per_day': 1.0, 'lead_time_hours': 8, 'change_failure_pct': 28.6, 'mttr_hours': 3.2}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Accumulate in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

statistics.median and comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Accumulate in a loop.

```python
def dora(deploys, days):
    leads = sorted(d - c for d, c, _, _ in deploys)
    n = len(leads)
    median = leads[n // 2] if n % 2 else (leads[n // 2 - 1] + leads[n // 2]) / 2
    fails = [(d, r) for d, _, f, r in deploys if f]
    mttr = 0
    for d, r in fails:
        mttr += r - d
    return {"deploys_per_day": round(len(deploys) / days, 2), "lead_time_hours": round(median, 1),
            "change_failure_pct": round(len(fails) / len(deploys) * 100, 1), "mttr_hours": round(mttr / len(fails), 1) if fails else 0}
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** statistics.median and comprehensions.

```python
def dora(deploys, days):
    fails = [r - d for d, _, f, r in deploys if f]
    return {"deploys_per_day": round(len(deploys) / days, 2), "lead_time_hours": round(statistics.median(d - c for d, c, _, _ in deploys), 1),
            "change_failure_pct": round(100 * len(fails) / len(deploys), 1), "mttr_hours": round(statistics.mean(fails), 1) if fails else 0}
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Accumulate in a loop. That costs O(n log n) time and O(n) space."

"I can do better: statistics.median and comprehensions. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/measure-delivery-dora-metrics)

<a id="q20"></a>
### Q20. Least-Privilege Token Permissions

🟡 Interview standard · *permissions:*

Each step needs certain `GITHUB_TOKEN` scopes (`NEEDS`). Return the smallest `permissions:` block for each job: for each scope, `'write'` if any step needs write, else `'read'`; sorted by scope. Granting only what's needed limits damage if a step is compromised.

**Examples**

- `jobs = {'test': ['checkout'], 'release': ['checkout', 'create release', 'push image'], 'review': ['read PR', 'checkout', 'comment on PR'], 'deploy': ['checkout', 'deploy to AWS', 'upload SARIF']}` → `{'test': {'contents': 'read'}, 'release': {'contents': 'write', 'packages': 'write'}, 'review': {'contents': 'read', 'pull-requests': 'write'}, 'deploy': {'contents': 'read', 'id-token': 'write', 'security-events': 'write'}}`

In [ ]:
NEEDS = {"checkout": {"contents": "read"}, "create release": {"contents": "write"}, "push image": {"packages": "write"},
         "comment on PR": {"pull-requests": "write"}, "read PR": {"pull-requests": "read"}, "deploy to AWS": {"id-token": "write"},
         "upload SARIF": {"security-events": "write"}}

def permissions(jobs):
    """
    Each step needs certain GITHUB_TOKEN scopes (NEEDS). Return the smallest
    permissions: block for each job: for each scope, 'write' if any step needs
    write, else 'read'; sorted by scope. Granting only what's needed limits
    damage if a step is compromised.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(permissions, [
    (({'test': ['checkout'], 'release': ['checkout', 'create release', 'push image'], 'review': ['read PR', 'checkout', 'comment on PR'], 'deploy': ['checkout', 'deploy to AWS', 'upload SARIF']},), {'test': {'contents': 'read'}, 'release': {'contents': 'write', 'packages': 'write'}, 'review': {'contents': 'read', 'pull-requests': 'write'}, 'deploy': {'contents': 'read', 'id-token': 'write', 'security-events': 'write'}}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Merge scope dicts with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

max() with write ranked above read.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Merge scope dicts with if-statements.

```python
def permissions(jobs):
    out = {}
    for job, steps in jobs.items():
        perms = {}
        for st in steps:
            for scope, level in NEEDS[st].items():
                if perms.get(scope) != "write":
                    perms[scope] = level
        out[job] = dict(sorted(perms.items()))
    return out
```

**Cost:** O(j · s) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** max() with write ranked above read.

```python
def permissions(jobs):
    rank = {"read": 0, "write": 1}
    def merge(steps):
        p = defaultdict(lambda: "read")
        for st in steps:
            for scope, lvl in NEEDS[st].items():
                p[scope] = max(p[scope], lvl, key=rank.get)
        return dict(sorted(p.items()))
    return {j: merge(s) for j, s in jobs.items()}
```

**Cost:** O(j · s) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Merge scope dicts with if-statements. That costs O(j · s) time and O(p) space."

"I can do better: max() with write ranked above read. That's O(j · s) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/cicd/least-privilege-token-permissions)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What should a CI pipeline check?</b></summary>

Formatting and linting, type checks, unit tests with coverage, a build of the artefact (package or image), and security scans of dependencies and the image. Slow integration and end-to-end tests can run on merge or nightly.

</details>

<details>
<summary><b>2. How do you speed up a slow pipeline?</b></summary>

Find the critical path, parallelise independent jobs, cache dependencies, shard tests, use path filters, cancel superseded runs and move slow suites out of the pull-request path.

</details>

<details>
<summary><b>3. How do you handle secrets in CI?</b></summary>

Use the platform's secret store, scope secrets to environments, prefer OIDC short-lived credentials, keep token permissions minimal, and never print secrets.

</details>

<details>
<summary><b>4. What is a matrix build?</b></summary>

Running the same job across combinations (Python versions, operating systems) to catch compatibility problems, with include and exclude to tailor the set.

</details>

<details>
<summary><b>5. How do you roll back a bad deploy?</b></summary>

Redeploy the previous image tag (or use the platform's rollout undo), which is why images are immutable and tagged by commit; fix forward afterwards.

</details>

<details>
<summary><b>6. Why pin actions to a SHA?</b></summary>

Tags can be moved to malicious code if an action's repository is compromised; a commit SHA can't change.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Your first CI workflow
- [ ] I can explain: Fast pipelines
- [ ] I can explain: Continuous delivery
- [ ] I can explain: Securing the pipeline

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Split Tests Across Shards
- [ ] [Q17](#q17) Generate a Changelog
- [ ] [Q18](#q18) Promote Through Environments
- [ ] [Q19](#q19) Measure Delivery: DORA Metrics
- [ ] [Q20](#q20) Least-Privilege Token Permissions

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.